# OPUS-MT en-tl — DIMER E2E English→Tagalog translation fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/marianmt-en-tl-translation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/marianmt-en-tl-translation-pipeline/blob/main/tutorials/marianmt_translation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Helsinki--NLP%2Fopus--mt--en--tl-ffcc4d?style=flat)](https://huggingface.co/Helsinki-NLP/opus-mt-en-tl) [![Upstream](https://img.shields.io/badge/Upstream-Helsinki--NLP%2FOPUS--MT--train-181717?style=flat&logo=github&logoColor=white)](https://github.com/Helsinki-NLP/OPUS-MT-train) [![arXiv](https://img.shields.io/badge/arXiv-1804.00344-b31b1b.svg)](https://arxiv.org/abs/1804.00344)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** batched English→Tagalog machine translation (EN→TL only) and bounded supervised fine-tuning of the last decoder blocks on a parallel corpus, using the pinned `Helsinki-NLP/opus-mt-en-tl` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/marianmt_translation_pipeline/`, at revision `b4b526274e66`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `e46e1761492cb6a6fb9515a72bb55ca654815ca5` (~299 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned OPUS-MT snapshot (a pickle checkpoint pinned by SHA-256 and loaded with `weights_only=True`), fetches the digest-pinned Tatoeba English–Tagalog corpus from OPUS (312 KB, no credential), filters and splits it into 1,200 / 200 / 300 disjoint training, validation and test pairs, translates three English sentences through the inference contract with an input manifest and a rejection probe, scores the frozen model on the test split with chrF and BLEU beside the copy-source baseline, runs a bounded fine-tuning of the last two decoder blocks on the training pairs with validation-chrF epoch selection, scores the held-out split again, translates new sentences with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify translation parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about two minutes of model time after the downloads.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own English–Tagalog pairs as a CSV (columns `id`, `source`, `target`), a JSON array or a JSONL file of `{{id, source, target}}` records. They pass through the same validation, seeded source-disjoint split, baselines, fine-tuning, held-out evaluation, inference, artifact export and reload-parity cells as the Tatoeba sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`Helsinki-NLP/opus-mt-en-tl` is a ~74 M-parameter OPUS-MT Marian transformer (6+6 layers, `d_model` 512) trained by the Helsinki-NLP group on the `opus+bt` corpus and released on 2020-02-26 — an **EN→TL only** model: English in, Tagalog out, and nothing else. At inference the encoder reads the English sentence once and the decoder emits one SentencePiece token per step until `</s>` or a step ceiling; **beam search with 4 beams** (the snapshot's `generation_config.json`) is the default decision rule and greedy decoding is available on request — there is no sampling and no seed. **The checkpoint is a pickle:** the only upstream weight file at this revision is `pytorch_model.bin` (no SafeTensors), so Section 3 re-hashes it against the inline SHA-256 manifest **before** it is opened, and the carried module then loads it with `use_safetensors=False, weights_only=True`, which makes `transformers` call `torch.load(weights_only=True)` — a restricted unpickler.

What this notebook adds to inference is **adaptation with references**. The dataset is real: the Tatoeba English–Tagalog sentence pairs as distributed by OPUS (release v2023-04-12, 8,785 pairs, CC BY 2.0 FR), fetched as one digest-pinned 312 KB zip and read member by member. It is the corpus the upstream README reports BLEU 26.6 / chrF 0.577 on; the pinned model was trained on OPUS data that predates this release, so the frozen model is already strong here — the frozen test BLEU in Section 6 lands within a point of the upstream figure — and the fine-tuning question is whether a small in-domain adaptation still moves held-out chrF and BLEU. Two metrics are implemented in the carried `metrics.py` (corpus chrF and BLEU, sacrebleu-style, not sacrebleu-identical), and the **copy-source baseline** — the English input submitted as the translation — shows where a system that does nothing sits. Nothing here is a quality claim about your domain: it is one seeded split of one corpus.

**Environment note:** `sacremoses` is not pinned and not installed, so `MarianTokenizer` prints one `Recommended: pip install sacremoses.` warning and uses the identity function as its source-side punctuation normaliser; every number below was produced in exactly that state.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, dataset and metrics modules guarantee; stage and digest-verify the immutable upstream snapshot — a pickle checkpoint pinned by SHA-256 and loaded with `weights_only=True`; fetch a digest-pinned parallel corpus and validate and split it without leakage; translate through the public API with explicit `max_new_tokens`/`num_beams` and read `stopped_by` and the token counts correctly; score the frozen model against references beside the copy-source baseline and read why chrF is the headline for Tagalog; run a bounded fine-tuning with explicit hyperparameters and validation-based epoch selection; evaluate on an independent test split; translate new sentences; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** Tagalog→English or any other direction, document-level translation with sentence splitting, instruction following or chat, sampling-based decoding, glossary or terminology control, source-side Moses punctuation normalisation (`sacremoses` is not installed), full-model or encoder fine-tuning, back-translation, COMET or any learned metric, and any claim that a Tatoeba split stands in for your domain. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is adequate: the build record measured 5.4 s to load and digest-verify the 299 MB snapshot, 11 s to translate the 300-sentence test split with 4 beams, and 56 s for the default two-epoch fine-tuning of the last two decoder blocks on 1,200 pairs. The pinned `torch==2.14.0` install and the 296 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what an encoder-decoder (seq2seq) model is; what beam search and greedy decoding do; why a translation can be fluent and wrong; what chrF and BLEU measure and why neither is a human judgement.
- **Data contract:** records are `{{id, source, target}}` — an English sentence and its Tagalog reference, each 1..4,000 characters, ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a dataset needs 8..20,000 records; sources are de-duplicated case-insensitively before splitting so the same English sentence never sits in two splits; during training only, sources and targets are truncated to 128 SentencePiece pieces (inference never truncates — it rejects). BYOD accepts CSV, JSON or JSONL in that shape.
- **Validation is structural, not linguistic:** nothing checks that a source is English, that a target is Tagalog, or that a pair is a faithful translation — a misaligned corpus is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — a proprietary translation memory is exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches one pinned object (`en-tl.txt.zip`, 312,459 bytes, SHA-256 `9abddc7d…`) from OPUS at `object.pouta.csc.fi` over HTTPS, refused on any mismatch before it is read; Tatoeba sentences are CC BY 2.0 FR (attribution: the Tatoeba contributors; redistribution: OPUS).
- **External access:** the Hugging Face Hub only, to fetch the pinned `Helsinki-NLP/opus-mt-en-tl` snapshot (~299 MB in total) at revision `e46e1761492c…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'sentencepiece==0.2.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'marianmt-en-tl-translation-pipeline',
    'repository_revision': 'b4b526274e66bdae56cc32ade1d5773e5de9e72c',
    'embedded_module': 'src/marianmt_translation_pipeline/pipeline.py',
    'embedded_modules': ['src/marianmt_translation_pipeline/metrics.py', 'src/marianmt_translation_pipeline/pipeline.py', 'src/marianmt_translation_pipeline/samples.py'],
    'module_sha256': '25fe96841d7aab736e6d83cf46df2b14da9f68a725e3b6c03e7fc215e09f3283',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/marianmt_translation_pipeline/` @ `b4b526274e66`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/marianmt_translation_pipeline/metrics.py`

In [ ]:
"""Translation quality metrics and the trivial baseline every adapted number is read against.

Two corpus-level scores are implemented here in plain Python so the notebook needs no scorer
dependency: **chrF** (character n-gram F-score, n = 1..6 with spaces removed, beta = 2, precision and
recall averaged over the orders — the sacrebleu ``chrF2`` recipe) and **BLEU** (corpus BLEU-4 with a
simple regex tokeniser and the brevity penalty; sacrebleu-*style*, not sacrebleu-identical, because the
``13a`` tokeniser is not reproduced). chrF is the headline: Tagalog is morphologically rich and
character n-grams reward partially correct affixed words that word-level BLEU counts as wrong. The
**copy-source baseline** submits the English input as the translation; it scores what a system that
does nothing scores, and it is where any translation model must start from.
"""

from __future__ import annotations

import collections
import math
import re
from collections.abc import Mapping, Sequence
from typing import Any

CHRF_MAX_ORDER = 6
CHRF_BETA = 2.0
BLEU_MAX_ORDER = 4
_TOKEN = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def _char_ngrams(text: str, order: int) -> collections.Counter:
    compact = text.replace(" ", "")
    return collections.Counter(compact[i : i + order] for i in range(len(compact) - order + 1))


def chrf(hypotheses: Sequence[str], references: Sequence[str]) -> float:
    """Corpus chrF (0..100) over aligned hypothesis/reference lists."""
    if len(hypotheses) != len(references) or not references:
        raise ValueError("hypotheses and references must be non-empty and equal in length")
    precisions, recalls = [], []
    for order in range(1, CHRF_MAX_ORDER + 1):
        matched = total_h = total_r = 0
        for hyp, ref in zip(hypotheses, references, strict=True):
            grams_h, grams_r = _char_ngrams(hyp, order), _char_ngrams(ref, order)
            matched += sum((grams_h & grams_r).values())
            total_h += sum(grams_h.values())
            total_r += sum(grams_r.values())
        precisions.append(matched / total_h if total_h else 0.0)
        recalls.append(matched / total_r if total_r else 0.0)
    precision = sum(precisions) / CHRF_MAX_ORDER
    recall = sum(recalls) / CHRF_MAX_ORDER
    if precision + recall == 0.0:
        return 0.0
    beta2 = CHRF_BETA**2
    return 100.0 * (1 + beta2) * precision * recall / (beta2 * precision + recall)


def tokenize(text: str) -> list[str]:
    """Lower-cased word/punctuation tokens for BLEU (a regex tokeniser, not Moses/13a)."""
    return _TOKEN.findall(text.lower())


def bleu(hypotheses: Sequence[str], references: Sequence[str]) -> float:
    """Corpus BLEU-4 (0..100) with uniform n-gram weights and the standard brevity penalty."""
    if len(hypotheses) != len(references) or not references:
        raise ValueError("hypotheses and references must be non-empty and equal in length")
    matched = [0] * BLEU_MAX_ORDER
    total = [0] * BLEU_MAX_ORDER
    hyp_len = ref_len = 0
    for hyp, ref in zip(hypotheses, references, strict=True):
        toks_h, toks_r = tokenize(hyp), tokenize(ref)
        hyp_len += len(toks_h)
        ref_len += len(toks_r)
        for order in range(1, BLEU_MAX_ORDER + 1):
            grams_h = collections.Counter(
                tuple(toks_h[i : i + order]) for i in range(len(toks_h) - order + 1)
            )
            grams_r = collections.Counter(
                tuple(toks_r[i : i + order]) for i in range(len(toks_r) - order + 1)
            )
            matched[order - 1] += sum((grams_h & grams_r).values())
            total[order - 1] += max(len(toks_h) - order + 1, 0)
    if min(matched) == 0 or min(total) == 0:
        return 0.0
    log_precision = sum(math.log(m / t) for m, t in zip(matched, total, strict=True)) / BLEU_MAX_ORDER
    brevity = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
    return 100.0 * brevity * math.exp(log_precision)


def translation_metrics(hypotheses: Sequence[str], references: Sequence[str]) -> dict[str, Any]:
    """chrF and BLEU plus the counts they were computed over."""
    return {
        "n": len(references),
        "chrf": chrf(hypotheses, references),
        "bleu": bleu(hypotheses, references),
        "hypothesis_chars": sum(len(h) for h in hypotheses),
        "reference_chars": sum(len(r) for r in references),
        "definitions": {
            "chrf": (
                "corpus chrF, character 1..6-grams, spaces removed, beta=2 "
                "(sacrebleu chrF2 recipe, own implementation)"
            ),
            "bleu": (
                "corpus BLEU-4, regex word/punctuation tokens lower-cased, brevity penalty "
                "(sacrebleu-style, not 13a)"
            ),
        },
    }


def copy_source_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Score the English source as if it were the translation: the do-nothing baseline."""
    if not records:
        raise ValueError("records must be non-empty")
    sources = [str(r["source"]) for r in records]
    targets = [str(r["target"]) for r in records]
    metrics = translation_metrics(sources, targets)
    metrics["baseline"] = "copy source (the English input submitted as the Tagalog output)"
    return metrics

**Module 2/3:** `src/marianmt_translation_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""English-to-Tagalog machine translation with the pinned ``Helsinki-NLP/opus-mt-en-tl`` checkpoint.

The class loads weights only from a digest-verified local snapshot (``weights/opus-mt-en-tl/``) or, when
explicitly allowed, from the Hugging Face Hub at the pinned revision. The upstream snapshot ships its
weights as ``pytorch_model.bin`` — a pickle, not SafeTensors — so the trust boundary is the SHA-256 in the
manifest (checked before the load) plus weights_only=True deserialization in ``transformers``. Two
task methods: ``translate`` — a batch of English strings in, one Tagalog string per input out — and
``adapt`` — bounded supervised fine-tuning of the last decoder layers on a validated parallel dataset,
with ``evaluate`` (chrF / BLEU against references, see ``metrics.py``) and a safetensors adapter artifact
that reloads against the pinned base. Direction is EN -> TL only; the checkpoint has no reverse direction.
"""

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "Helsinki-NLP/opus-mt-en-tl"
MODEL_REVISION = "e46e1761492cb6a6fb9515a72bb55ca654815ca5"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "opus-mt-en-tl"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHT_FILE = "pytorch_model.bin"  # the only upstream weight file at this revision: a pickle, digest-pinned

SOURCE_LANG = "en"  # tokenizer_config.json source_lang
TARGET_LANG = "tl"  # tokenizer_config.json target_lang
MAX_INPUT_TOKENS = 512  # max_position_embeddings in the snapshot config.json; longer inputs rejected, not cut
MAX_NEW_TOKENS = 512  # ceiling on decoder steps per call (config.json / generation_config.json max_length)
DEFAULT_MAX_NEW_TOKENS = 128
MAX_TEXT_CHARS = 4_000  # pre-tokenisation guard per input string
MAX_BATCH = 16  # texts per translate() call
MAX_NUM_BEAMS = 8
DEFAULT_NUM_BEAMS = 4  # num_beams in the snapshot generation_config.json
DECISION_RULE = (
    "beam search over whole sequences (num_beams=4 by default, from generation_config.json); greedy argmax "
    "per step when num_beams=1; no sampling; decoding stops at </s> or max_new_tokens"
)
WEIGHT_SHA256 = (
    "e418d573a717b2c81eaa3a80c8eb68f203c14c6b734dd36d41708c1369d0eb44"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 74_037_760
DECODER_LAYERS = 6  # config.json decoder_layers
DEFAULT_TRAINABLE_DECODER_LAYERS = 2  # the last two decoder blocks (8,408,064 parameters)
MAX_TRAIN_TOKENS = 128  # source/target truncation ceiling during adaptation (never at inference)
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_REFERENCES = 50  # below this a referenced score is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.marianmt-en-tl.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


INPUT_SCHEMA: dict[str, Any] = {
    "input": "sequence of non-empty English str (a sentence or short passage each); one Tagalog str each",
    "direction": f"{SOURCE_LANG}->{TARGET_LANG} only",
    "batch": [1, MAX_BATCH],
    "text_chars": [1, MAX_TEXT_CHARS],
    "input_tokens": [1, MAX_INPUT_TOKENS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "num_beams": [1, MAX_NUM_BEAMS],
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "Marian normalisation + SentencePiece encoding with source.spm (no truncation: an input over "
        "MAX_INPUT_TOKENS is rejected with a ValueError naming the count, never cut); batch padded to the "
        "longest input"
    ),
}


def _check_inputs(texts: Any, max_new_tokens: Any, num_beams: Any) -> list[str]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the texts as a list.

    The encoder-token ceiling is not checked here because it needs the loaded tokenizer;
    ``_check_input_tokens`` applies it inside the pipeline once the counts are known.
    """
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a sequence of str, not a single string")
    if not 1 <= len(texts) <= MAX_BATCH:
        raise ValueError(f"texts must hold 1..MAX_BATCH={MAX_BATCH} items, got {len(texts)}")
    clean = []
    for i, text in enumerate(texts):
        if not isinstance(text, str):
            raise TypeError(f"texts[{i}] must be str, got {type(text).__name__}")
        if not text.strip():
            raise ValueError(f"texts[{i}] is empty")
        if len(text) > MAX_TEXT_CHARS:
            raise ValueError(f"texts[{i}] has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
        clean.append(text)
    for name, value, ceiling in (
        ("max_new_tokens", max_new_tokens, MAX_NEW_TOKENS),
        ("num_beams", num_beams, MAX_NUM_BEAMS),
    ):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} must be an int")
        if not 1 <= value <= ceiling:
            raise ValueError(f"{name} must be between 1 and {ceiling}, got {value}")
    return clean


def _check_input_tokens(counts: Sequence[int]) -> list[int]:
    """The encoder-token ceiling, applied once the tokenizer has counted every input."""
    for i, n_input in enumerate(counts):
        if n_input > MAX_INPUT_TOKENS:
            raise ValueError(
                f"texts[{i}] is {n_input} tokens; ceiling is MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}"
            )
    return list(counts)


def validate_inputs(
    texts: Sequence[str],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    num_beams: int = DEFAULT_NUM_BEAMS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``translate`` would: both route through
    ``_check_inputs``. The encoder-token ceiling (``MAX_INPUT_TOKENS``) needs the loaded tokenizer and
    is enforced inside ``translate``, which reports ``input_tokens`` per item.
    """
    checked = _check_inputs(texts, max_new_tokens, num_beams)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"input{i:02d}", "chars": len(text), "words": len(text.split())}
            for i, text in enumerate(checked)
        ],
        "max_new_tokens": max_new_tokens,
        "num_beams": num_beams,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], references: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    Without ``references`` the verdict is ``not-measurable`` (EVAL9): manufacturing a number from a
    proxy such as length ratio would misrepresent a plumbing check as a quality measurement. With
    references — one per translation — chrF and BLEU are computed by ``metrics.py`` and the verdict is
    ``measured`` (``measured-small-sample`` below ``MIN_SCORED_REFERENCES`` items, which carries no
    dispersion estimate).
    """
    items = result.get("translations", [])
    rule = result.get("generation", {}).get("decision_rule", DECISION_RULE)
    supplied = references is not None
    if supplied:
        pass  # standalone rewrite (build_notebook.py): `from .metrics import translation_metrics` removed — names are kernel globals defined by the carried modules

        if len(references) != len(items):
            raise ValueError("references must have one entry per translation")
        scored = translation_metrics(
            [str(item.get("text", "")) for item in items], [str(r) for r in references]
        )
        return {
            "task": f"machine translation {SOURCE_LANG}->{TARGET_LANG}",
            "score_semantics": (
                "chrF and BLEU are corpus-level agreement with the supplied references (own "
                "implementations, see "
                "metrics.py); the pipeline itself emits no probability, confidence or score, and "
                f"{rule} produces some token at every step with no acceptance threshold"
            ),
            "sample_kind": sample_kind,
            "n_inputs": len(items),
            "n_generated_tokens": int(sum(int(item.get("generated_tokens", 0)) for item in items)),
            "metrics": [{"name": "chrf", "value": scored["chrf"]}, {"name": "bleu", "value": scored["bleu"]}],
            "baselines": [],
            "verdict": "measured" if len(items) >= MIN_SCORED_REFERENCES else "measured-small-sample",
            "reason": (
                f"scored {len(items)} referenced sentences; "
                + (
                    "a sample this small carries no dispersion estimate"
                    if len(items) < MIN_SCORED_REFERENCES
                    else "one seeded holdout, no dispersion estimate"
                )
            ),
            "needs": "references from the deployment domain over enough sentences to state a dispersion",
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }
    return {
        "task": f"machine translation {SOURCE_LANG}->{TARGET_LANG}",
        "score_semantics": (
            "the pipeline emits no probability, confidence or score: generated_tokens, input_tokens and "
            f"stopped_by are counts and flags, and {rule} "
            "produces some token at every step with no minimum-probability cut-off and no shipped acceptance "
            "threshold"
        ),
        "sample_kind": sample_kind,
        "n_inputs": len(items),
        "n_generated_tokens": int(sum(int(item.get("generated_tokens", 0)) for item in items)),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "a translation has no ground truth here; the evaluated sample has no reference translations"
        ),
        "needs": (
            "reference Tagalog translations from the deployment domain, one or more per source sentence, "
            "over enough sentences to state a dispersion, passed as `references` (scored with the chrF/BLEU "
            "helpers in metrics.py) or through evaluate() on {id, source, target} records; the upstream "
            "README reports BLEU 26.6 / chrF 0.577 on Tatoeba.en.tl — an upstream claim reproduced only as "
            "far as this repository's own Tatoeba split allows; exclude or re-run outputs whose stopped_by "
            "is max_new_tokens; no proxy such as length ratio substitutes for references"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class MarianMTTranslationPipeline:
    """``_runner(texts, max_new_tokens, num_beams)`` -> list of ``(translation, generated_tokens,
    stopped_by)``, one per input; ``_count_tokens(text)`` -> encoder token count incl. EOS. Both
    injectable so tests run offline."""

    _runner: Callable[[list[str], int, int], list[tuple[str, int, str]]]
    _count_tokens: Callable[[str], int]
    device: str = "cpu"
    source: str = "injected"
    adapter: dict[str, Any] | None = None
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> MarianMTTranslationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, kwargs, source = str(root), dict(local_files_only=True), "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, dict(revision=MODEL_REVISION), "hf-hub"
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import MarianMTModel, MarianTokenizer

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = MarianTokenizer.from_pretrained(location, trust_remote_code=False, **kwargs)
        # Trust boundary (MOD12): the upstream weight file is a pickle (pytorch_model.bin). Its SHA-256 was
        # checked against the manifest above; use_safetensors=False names that fact, and weights_only=True
        # makes transformers deserialise with weights_only=True, which refuses arbitrary objects.
        model = MarianMTModel.from_pretrained(
            location,
            dtype=torch.float32,
            trust_remote_code=False,
            use_safetensors=False,
            weights_only=True,
            **kwargs,
        )
        model = model.to(resolved_device).eval()
        eos_id, pad_id = model.config.eos_token_id, model.config.pad_token_id

        def count_tokens(text: str) -> int:
            return len(tokenizer(text, truncation=False)["input_ids"])

        def runner(texts: list[str], max_new_tokens: int, num_beams: int) -> list[tuple[str, int, str]]:
            enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=False).to(resolved_device)
            with torch.inference_mode():
                out = model.generate(
                    **enc, max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False
                )
            results = []
            for row in out.tolist():
                content = [t for t in row if t not in (eos_id, pad_id)]
                stopped_by = "eos" if eos_id in row else "max_new_tokens"
                results.append(
                    (tokenizer.decode(content, skip_special_tokens=True), len(content), stopped_by)
                )
            return results

        return cls(runner, count_tokens, resolved_device, source, _model=model, _tokenizer=tokenizer)

    def _validate(self, texts: Any, max_new_tokens: Any, num_beams: Any) -> tuple[list[str], list[int]]:
        clean = _check_inputs(texts, max_new_tokens, num_beams)
        return clean, _check_input_tokens([self._count_tokens(text) for text in clean])

    def translate(
        self,
        texts: Sequence[str],
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        num_beams: int = DEFAULT_NUM_BEAMS,
    ) -> dict[str, Any]:
        """Translate a batch of English texts to Tagalog; one output per input, in order."""
        clean, counts = self._validate(texts, max_new_tokens, num_beams)
        generated = self._runner(clean, max_new_tokens, num_beams)
        if len(generated) != len(clean):
            raise RuntimeError(f"runner returned {len(generated)} outputs for {len(clean)} inputs")
        translations = []
        for text, n_input, (output, n_generated, stopped_by) in zip(clean, counts, generated, strict=True):
            if not isinstance(output, str) or not isinstance(n_generated, int):
                raise RuntimeError("runner must return (str, int, str) per input")
            translations.append(
                {
                    "source": text,
                    "text": output,
                    "input_tokens": n_input,
                    "generated_tokens": n_generated,
                    "stopped_by": stopped_by,
                }
            )
        return {
            "translations": translations,
            "n": len(translations),
            "direction": f"{SOURCE_LANG}->{TARGET_LANG}",
            "generation": {
                "max_new_tokens": max_new_tokens,
                "num_beams": num_beams,
                "do_sample": False,
                "decision_rule": DECISION_RULE,
            },
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation -----------------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        batch_size: int = MAX_BATCH,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        num_beams: int = DEFAULT_NUM_BEAMS,
    ) -> dict[str, Any]:
        """Translate every record's source and score the outputs against its target (chrF, BLEU)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import translation_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        if not isinstance(batch_size, int) or not 1 <= batch_size <= MAX_BATCH:
            raise ValueError(f"batch_size must be an int in 1..{MAX_BATCH}")
        started = time.perf_counter()
        hypotheses: list[str] = []
        truncated = 0
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            result = self.translate(
                [r["source"] for r in batch], max_new_tokens=max_new_tokens, num_beams=num_beams
            )
            for item in result["translations"]:
                hypotheses.append(item["text"])
                truncated += item["stopped_by"] == "max_new_tokens"
        metrics = translation_metrics(hypotheses, [r["target"] for r in checked])
        metrics.update(
            {
                "hit_token_ceiling": truncated,
                "generation": {"max_new_tokens": max_new_tokens, "num_beams": num_beams, "do_sample": False},
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _trainable_names(self, trainable_decoder_layers: int) -> list[str]:
        if (
            not isinstance(trainable_decoder_layers, int)
            or not 1 <= trainable_decoder_layers <= DECODER_LAYERS
        ):
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{DECODER_LAYERS}")
        model, _ = self._require_model()
        first = DECODER_LAYERS - trainable_decoder_layers
        prefixes = tuple(f"model.decoder.layers.{k}." for k in range(first, DECODER_LAYERS))
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 2,
        lr: float = 1e-4,
        batch_size: int = 16,
        trainable_decoder_layers: int = DEFAULT_TRAINABLE_DECODER_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning on a validated parallel dataset.

        Only the last `trainable_decoder_layers` decoder blocks train (2 by default: 8,408,064 of
        74,037,760 parameters; the encoder, the shared embeddings and the earlier decoder blocks stay
        frozen). Teacher-forced cross-entropy on the Tagalog target (label smoothing 0), AdamW at a fixed
        learning rate with gradient clipping at 1.0, sources and targets truncated to MAX_TRAIN_TOKENS
        SentencePiece pieces **during training only**. Epoch 0 records the frozen model's validation chrF;
        the epoch with the highest validation chrF is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-2):
            raise ValueError("lr must be in (0, 1e-2]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        names = self._trainable_names(trainable_decoder_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        import torch

        torch.manual_seed(seed)
        model, tokenizer = self._require_model()
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = torch.device(self.device)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {
                k: v
                for k, v in self.evaluate(val_checked).items()
                if k in ("chrf", "bleu", "n", "hit_token_ceiling")
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_chrf = entry["val"]["chrf"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [train_checked[i] for i in order[start : start + batch_size]]
                    encoded = tokenizer(
                        [r["source"] for r in batch],
                        text_target=[r["target"] for r in batch],
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=MAX_TRAIN_TOKENS,
                    )
                    labels = encoded["labels"].clone()
                    labels[labels == tokenizer.pad_token_id] = -100
                    out = model(
                        input_ids=encoded["input_ids"].to(device),
                        attention_mask=encoded["attention_mask"].to(device),
                        labels=labels.to(device),
                    )
                    optimiser.zero_grad(set_to_none=True)
                    out.loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(out.loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                current = entry["val"]["chrf"] if entry["val"] else math.inf
                if current > best_chrf or not entry["val"]:
                    best_chrf = current
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the base
            # exactly as it was, with every parameter frozen again.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_decoder_layers": trainable_decoder_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation chrF" if val_checked else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "max_train_tokens": MAX_TRAIN_TOKENS,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted decoder tensors as safetensors with a manifest naming the pinned base."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:
        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported
        format and version, the pinned base (id, revision, weight file, digest), exactly one file entry
        named `adapter.safetensors` that resolves inside the artifact directory, and a recorded
        `trainable_decoder_layers` in range. Nothing is deserialised here. The digest check that follows
        detects corruption or drift of the weights relative to the adjacent manifest; it is not
        authenticity against an actor who can replace both files."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not the supported "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", WEIGHT_FILE) != WEIGHT_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        layers = adapter.get("trainable_decoder_layers") if isinstance(adapter, Mapping) else None
        if isinstance(layers, bool) or not isinstance(layers, int):
            raise ValueError("artifact manifest does not record an integer trainable_decoder_layers")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        # The exact tensor set the recorded configuration implies — no subset, no extra, no other layer.
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_decoder_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("model.decoder.layers."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable decoder tensor of the base model"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> MarianMTTranslationPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/marianmt_translation_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Parallel-corpus dataset contract for fine-tuning: the pinned Tatoeba en–tl sample, validation,
seeded splitting, BYOD loaders and CSV export.

The default dataset is **real**: the English–Tagalog sentence pairs of the Tatoeba corpus as
distributed by OPUS (release v2023-04-12, Moses format, 8,785 pairs, CC BY 2.0 FR with attribution to
the Tatoeba contributors). One 312 KB zip is fetched from OPUS's object store, refused on any byte-size
or SHA-256 mismatch, and read member by member (no `extractall`). Pairs are filtered to 3..200
characters a side and de-duplicated on the lower-cased English side, so a source sentence can appear in
only one split; the split is a seeded shuffle with fixed train/validation/test sizes.

Why Tatoeba: it is the corpus the upstream README reports BLEU 26.6 / chrF 0.577 on — the pinned model
was trained on OPUS data that predates this 2023 release, so the frozen model is strong here and the
fine-tuning question is whether a small in-domain adaptation still moves held-out chrF and BLEU (the
build record says it does). A record is ``{id, source, target}``: an English sentence and its Tagalog
reference.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_TEXT_CHARS, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "Tatoeba"
CORPUS_RELEASE = "v2023-04-12"
CORPUS_URL = "https://object.pouta.csc.fi/OPUS-Tatoeba/v2023-04-12/moses/en-tl.txt.zip"
CORPUS_BYTES = 312_459
CORPUS_SHA256 = "9abddc7d2fcea307e8582285617d1a10246fdb9050b6c6345243cb4b9991f40b"
CORPUS_MEMBERS = {"source": "Tatoeba.en-tl.en", "target": "Tatoeba.en-tl.tl"}
CORPUS_LICENSE = "CC BY 2.0 FR (Tatoeba contributors; OPUS redistribution)"
CORPUS_PAIRS = 8_785
DEFAULT_CACHE_DIR = Path("weights") / "tatoeba-en-tl"
MIN_PAIR_CHARS = 3
MAX_PAIR_CHARS = 200
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 1_200, "validation": 200, "test": 300}
MIN_RECORDS = 8
MAX_RECORDS = 20_000
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> bytes:
    """Return the pinned corpus zip bytes from the cache or the OPUS object store, digest-verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / "en-tl.txt.zip"
    if local.is_file():
        data = local.read_bytes()
        if len(data) == CORPUS_BYTES and _sha256_bytes(data) == CORPUS_SHA256:
            return data
    if fetcher is not None:
        data = fetcher(CORPUS_URL)
    else:
        with urllib.request.urlopen(CORPUS_URL, timeout=120) as response:  # noqa: S310 (pinned https URL)
            data = response.read()
    if len(data) != CORPUS_BYTES or _sha256_bytes(data) != CORPUS_SHA256:
        raise ValueError(
            f"corpus zip: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
            f"pinned {CORPUS_BYTES} / {CORPUS_SHA256[:16]}…"
        )
    local.write_bytes(data)
    return data


def read_corpus_pairs(data: bytes) -> list[tuple[str, str]]:
    """Aligned (English, Tagalog) lines from the Moses members, read without extracting to disk."""
    archive = zipfile.ZipFile(io.BytesIO(data))
    names = set(archive.namelist())
    for member in CORPUS_MEMBERS.values():
        if member not in names:
            raise ValueError(f"corpus zip is missing member {member}")
    source = archive.read(CORPUS_MEMBERS["source"]).decode("utf-8").splitlines()
    target = archive.read(CORPUS_MEMBERS["target"]).decode("utf-8").splitlines()
    if len(source) != len(target):
        raise ValueError(f"corpus members are not aligned: {len(source)} vs {len(target)} lines")
    return [(s.strip(), t.strip()) for s, t in zip(source, target, strict=True)]


def filter_pairs(pairs: Sequence[tuple[str, str]]) -> list[tuple[str, str]]:
    """Keep pairs with both sides in MIN_PAIR_CHARS..MAX_PAIR_CHARS, one pair per lower-cased source."""
    seen: set[str] = set()
    kept = []
    for source, target in pairs:
        if not (
            MIN_PAIR_CHARS <= len(source) <= MAX_PAIR_CHARS
            and MIN_PAIR_CHARS <= len(target) <= MAX_PAIR_CHARS
        ):
            continue
        key = source.lower()
        if key in seen:
            continue
        seen.add(key)
        kept.append((source, target))
    return kept


def build_sample_dataset(
    pairs: Sequence[tuple[str, str]], *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] | None = None
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of the filtered pairs cut into the named split sizes; ids carry the split name."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    kept = filter_pairs(pairs)
    total = sum(sizes.values())
    if total > len(kept):
        raise ValueError(f"requested {total} records but only {len(kept)} filtered pairs are available")
    order = list(range(len(kept)))
    random.Random(seed).shuffle(order)
    splits: dict[str, list[dict[str, Any]]] = {}
    cursor = 0
    for name, size in sizes.items():
        splits[name] = [
            {"id": f"{name}-{i:04d}", "source": kept[j][0], "target": kept[j][1]}
            for i, j in enumerate(order[cursor : cursor + size])
        ]
        cursor += size
    return splits


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus_pairs(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/source/target")
    for key in ("id", "source", "target"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid, source, target = record["id"], record["source"], record["target"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    for key, value in (("source", source), ("target", target)):
        if not isinstance(value, str):
            raise ValueError(f"{label}: {key} must be a string")
        if not value.strip():
            raise ValueError(f"{label}: {key} is empty")
        if len(value) > MAX_TEXT_CHARS:
            raise ValueError(
                f"{label}: {key} has {len(value)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}"
            )
    return {"id": rid, "source": source.strip(), "target": target.strip()}


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a parallel dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, source, target} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    sources: set[str] = set()
    identical = 0
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        sources.add(item["source"].lower())
        identical += item["source"] == item["target"]
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_sources": len(sources),
        "identical_pairs": identical,
        "source_chars": {
            "min": min(len(r["source"]) for r in checked),
            "max": max(len(r["source"]) for r in checked),
        },
        "target_chars": {
            "min": min(len(r["target"]) for r in checked),
            "max": max(len(r["target"]) for r in checked),
        },
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], r["source"], r["target"]] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no lower-cased English source appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record["source"]).lower()
            if key in seen and seen[key] != name:
                raise ValueError(f"source {record['source']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating sources."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = record["source"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {
        "test": unique[:n_test],
        "validation": unique[n_test : n_test + n_val],
        "train": unique[n_test + n_val :],
    }
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, source, target}` records from CSV (columns id, source, target), a JSON array or JSONL."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".csv":
        rows = list(csv.DictReader(io.StringIO(text)))
        missing = {"id", "source", "target"} - set(rows[0].keys() if rows else set())
        if missing:
            raise ValueError(f"CSV is missing columns {sorted(missing)}")
        return [{"id": r["id"], "source": r["source"], "target": r["target"]} for r in rows]
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    raise ValueError("BYOD datasets must be .csv, .json or .jsonl")


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "source", "target"])
        writer.writeheader()
        for record in records:
            writer.writerow({"id": record["id"], "source": record["source"], "target": record["target"]})
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `e46e1761492c…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `MarianMTTranslationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "opus-mt-en-tl",
  "modelId": "Helsinki-NLP/opus-mt-en-tl",
  "revision": "e46e1761492cb6a6fb9515a72bb55ca654815ca5",
  "files": [
    {
      "path": "README.md",
      "bytes": 839,
      "sha256": "debb5b41e6d8c18590466ce0c3b9bcfc3634f14454cac8f66b71dd3dcb36a58e"
    },
    {
      "path": "config.json",
      "bytes": 1361,
      "sha256": "f7c975ff86205d2e19e4a2fa93dd8cd124a707ab567f4044173fd9cda895d1aa"
    },
    {
      "path": "generation_config.json",
      "bytes": 293,
      "sha256": "2e6d06b221eee26979f89f53735306f4e3d927808cc61a062218936729c4329a"
    },
    {
      "path": "pytorch_model.bin",
      "bytes": 296434867,
      "sha256": "e418d573a717b2c81eaa3a80c8eb68f203c14c6b734dd36d41708c1369d0eb44"
    },
    {
      "path": "source.spm",
      "bytes": 826681,
      "sha256": "feaf3e0cf579e3daeb6bac5dc94b906efed40e89b544d269cec9bdbb4f05f8e1"
    },
    {
      "path": "target.spm",
      "bytes": 834725,
      "sha256": "e26d20a4aae1e82aaab382ff47ad8b587d03d1087e295d4c51ff71d63427e059"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 42,
      "sha256": "5ee0a3aea386b802c612ef5cce6f77fa4271ccb01b64dce0e4aca14725203ee1"
    },
    {
      "path": "vocab.json",
      "bytes": 1324714,
      "sha256": "43bf29e5d430061afa7d978d68c09683dc95650959607ae74eaaf5471285df57"
    }
  ],
  "totalBytes": 299423522
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = MarianMTTranslationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Parallel corpus, validation and split

`fetch_sample_dataset` downloads the pinned Tatoeba en–tl zip from OPUS (or reads it from the cache), refuses a byte-size or SHA-256 mismatch before the archive is opened, reads the two Moses members without extracting to disk, keeps pairs with both sides in 3..200 characters, drops repeated English sources case-insensitively, and cuts a seeded shuffle into 1,200 training, 200 validation and 300 test records. `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no English source appears in two splits, and the training split is written to `outputs/marianmt_translation_train.csv` in the shape BYOD expects.

Look for: 8,785 raw pairs, three digests, splits 1,200 / 200 / 300, zero identical pairs, and four refusal probes — a duplicate id, an empty target, a missing field and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import hashlib
import io
import json

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_pairs = len(records)
else:
    corpus_bytes = fetch_corpus(cache_dir='weights/tatoeba-en-tl')
    raw_pairs = len(read_corpus_pairs(corpus_bytes))
    splits = build_sample_dataset(read_corpus_pairs(corpus_bytes), seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} en-tl {CORPUS_RELEASE} via OPUS ({CORPUS_LICENSE})'
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
write_dataset_csv(train_records, 'outputs/marianmt_translation_train.csv')
print({'data_source': data_source, 'raw_pairs': raw_pairs, 'splits': disjoint, 'corpus_sha256': CORPUS_SHA256[:16] + '...'})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_sources': manifest['unique_sources'], 'identical_pairs': manifest['identical_pairs'], 'source_chars': manifest['source_chars'], 'digest': manifest['digest'][:16] + '...'}})
print({'example': train_records[0]})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty target': [{**train_records[0], 'target': ' '}, *train_records[1:8]],
    'missing field': [{'id': r['id'], 'source': r['source']} for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Translate through the inference contract

Before any adaptation, the inference contract is exercised as it always was: `validate_inputs` applies exactly the checks `translate` applies (batch size 1..`MAX_BATCH`, non-empty strings under `MAX_TEXT_CHARS`, `max_new_tokens` and `num_beams` within their ceilings) and returns an input manifest; the encoder-token ceiling `MAX_INPUT_TOKENS` needs the real tokenizer and is enforced inside `translate`, which **rejects with a `ValueError` naming the count, never silently cuts**. `translate` returns one entry per input, in order, with `source`, `text`, `input_tokens`, `generated_tokens` and `stopped_by` (`eos` when the model ended the sequence itself, `max_new_tokens` when it hit the ceiling and the output is cut). **Score semantics:** the pipeline emits **no probability, confidence or score of any kind** — the counts are counts, and beam search always produces some token. The three sentences are the ones the repository's original smoke used; whether their translations are *good* is what Section 6 measures on 300 referenced sentences, not what these three can tell you.

In [ ]:
import time

GEN_MAX_NEW_TOKENS = 128  # @param {type:"integer"}
NUM_BEAMS = 4  # @param {type:"integer"}

texts = ['The house is wonderful.', 'Good morning to all of you.', 'Where is the nearest hospital?']
item_ids = [f'input{index:02d}' for index in range(len(texts))]
ceilings = {'MAX_BATCH': MAX_BATCH, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_INPUT_TOKENS': MAX_INPUT_TOKENS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'MAX_NUM_BEAMS': MAX_NUM_BEAMS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DEFAULT_NUM_BEAMS': DEFAULT_NUM_BEAMS}
print(ceilings)
print({'decision_rule': DECISION_RULE, 'direction': f'{SOURCE_LANG}->{TARGET_LANG}'})
input_manifest = validate_inputs(texts, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, names=item_ids)
try:
    validate_inputs(texts, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=MAX_NUM_BEAMS + 1)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'num-beams-ceiling-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/marianmt_translation_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
started = time.perf_counter()
result = pipe.translate(texts, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
elapsed = time.perf_counter() - started
results = [{'id': item_id, **item} for item_id, item in zip(item_ids, result['translations'], strict=True)]
for r in results:
    print(f"{r['id']} {r['input_tokens']} -> {r['generated_tokens']} tokens, stopped_by={r['stopped_by']}: {r['source']} => {r['text']}")
checks = {
    'one_result_per_input_in_order': [r['source'] for r in results] == list(texts),
    'input_within_ceiling': all(r['input_tokens'] <= MAX_INPUT_TOKENS for r in results),
    'direction_is_en_to_tl': result['direction'] == f'{SOURCE_LANG}->{TARGET_LANG}',
    'settings_echoed': result['generation']['max_new_tokens'] == GEN_MAX_NEW_TOKENS and result['generation']['num_beams'] == NUM_BEAMS and result['generation']['do_sample'] is False,
}
if not all(checks.values()):
    raise RuntimeError(f'translate output failed a sanity check: {checks}')
print({'batch_seconds': round(elapsed, 3), 'checks': checks, 'hit_token_ceiling': [r['id'] for r in results if r['stopped_by'] == 'max_new_tokens'], 'findings': len(input_manifest['findings'])})

## 6. Baselines and the frozen model's score on the test split

Two numbers frame the adaptation. The **copy-source baseline** submits every English test sentence as its own translation and scores it against the Tagalog reference: what a system that does nothing gets, and a reminder that chrF rewards shared characters (names, numbers, punctuation) even across languages. The **frozen model** translates the 300 test sentences with the settings from Section 5 and is scored with the same two metrics: corpus **chrF** (character 1..6-grams, spaces removed, beta 2 — the headline, because Tagalog's affixation makes word-level matching harsh) and corpus **BLEU-4** (regex-tokenised, with the brevity penalty; sacrebleu-style, not sacrebleu-identical). Expect the frozen BLEU to land within a point of the upstream README's 26.6 on Tatoeba — a sanity check that the pinned weights, the tokenizer and the decoding path are the upstream ones — and read `hit_token_ceiling`, the count of outputs cut at `max_new_tokens`, before trusting any score.

In [ ]:
baseline_copy = copy_source_baseline(test_records)
print({'copy_source_baseline': {'chrf': round(baseline_copy['chrf'], 2), 'bleu': round(baseline_copy['bleu'], 2), 'n': baseline_copy['n']}})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
print({'frozen_model_test': {'chrf': round(frozen_test['chrf'], 2), 'bleu': round(frozen_test['bleu'], 2), 'n': frozen_test['n'], 'hit_token_ceiling': frozen_test['hit_token_ceiling']}, 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
shown = pipe.translate([r['source'] for r in test_records[:3]], max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)['translations']
for record, item in zip(test_records[:3], shown, strict=True):
    print({'source': record['source'], 'frozen': item['text'], 'reference': record['target']})
assert frozen_test['chrf'] > baseline_copy['chrf']

## 7. Bounded fine-tuning of the last decoder blocks

`pipe.adapt` trains only the last `TRAINABLE_DECODER_LAYERS` decoder blocks — two by default, 8,408,064 of 74,037,760 parameters; the encoder, the shared embeddings and the earlier decoder blocks stay frozen — with teacher-forced cross-entropy on the Tagalog target, AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler. Sources and targets are truncated to 128 SentencePiece pieces **during training only**. Epoch 0 records the frozen model's validation chrF and BLEU; every epoch is scored on the validation split, and the epoch with the highest validation chrF is kept.

Watch validation chrF rise a few points over two epochs (about a minute on CPU). The build record's counter-examples: training the whole decoder (25 M parameters) or the whole model (74 M) gained nothing beyond the last two blocks on this corpus while costing more time and a far larger artifact.

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 16  # @param {type:"integer"}
TRAINABLE_DECODER_LAYERS = 2  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_chrf'] = round(entry['val']['chrf'], 2)
        row['val_bleu'] = round(entry['val']['bleu'], 2)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_decoder_layers=TRAINABLE_DECODER_LAYERS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or epoch selection, and no English source in it appears in the training split. The adapted model is scored exactly as the frozen model was in Section 6, and the three numbers are put side by side. Look for a chrF gain of a few points and a BLEU gain of several — the cell asserts the adapted chrF is above the frozen chrF — and for the same three sentences translated by the adapted model. Three hundred sentences from one seeded split of one corpus give no dispersion estimate; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and an in-domain gain on Tatoeba says nothing about your domain until you measure it there.

In [ ]:
adapted_test = pipe.evaluate(test_records, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
adapted_val = pipe.evaluate(val_records, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
comparison = {
    'chrf': {'copy_source': round(baseline_copy['chrf'], 2), 'frozen': round(frozen_test['chrf'], 2), 'adapted': round(adapted_test['chrf'], 2)},
    'bleu': {'copy_source': round(baseline_copy['bleu'], 2), 'frozen': round(frozen_test['bleu'], 2), 'adapted': round(adapted_test['bleu'], 2)},
    'delta_vs_frozen': {'chrf': round(adapted_test['chrf'] - frozen_test['chrf'], 2), 'bleu': round(adapted_test['bleu'] - frozen_test['bleu'], 2)},
}
for metric, row in comparison.items():
    print({metric: row})
shown_adapted = pipe.translate([r['source'] for r in test_records[:3]], max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)['translations']
for record, item in zip(test_records[:3], shown_adapted, strict=True):
    print({'source': record['source'], 'adapted': item['text'], 'reference': record['target']})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'generation': frozen_test['generation'],
    'baselines': {'copy_source': baseline_copy},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/marianmt_translation_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['chrf'] > frozen_test['chrf']
print({'report': 'outputs/marianmt_translation_evaluation_report.json'})

## 9. Translate new sentences, export the adapter and reload it

Six sentences that were in none of the splits are translated by the adapted model through the same `translate` contract as Section 5, and scored with `evaluation_report` — the inference-stage helper, which now returns a `measured-small-sample` verdict when references are supplied, because six sentences carry no dispersion estimate.

`pipe.save_artifact` writes the trained tensors — the last two decoder blocks, about 34 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base pickle checkpoint, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `MarianMTTranslationPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, refuses any tensor that is not an adaptable decoder tensor, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical translations (VER4).

In [ ]:
import csv
import shutil

held_out = [r for r in read_corpus_pairs(corpus_bytes) if r[0].lower() not in {x['source'].lower() for part in splits.values() for x in part}][:6] if not USE_BYOD else [(r['source'], r['target']) for r in test_records[:6]]
new_records = [{'id': f'new-{i:02d}', 'source': s, 'target': t} for i, (s, t) in enumerate(held_out)]
new_result = pipe.translate([r['source'] for r in new_records], max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
new_report = evaluation_report(new_result, [r['target'] for r in new_records], sample_kind='six unseen Tatoeba pairs' if not USE_BYOD else 'BYOD test records')
for record, item in zip(new_records, new_result['translations'], strict=True):
    print({'id': record['id'], 'source': record['source'], 'adapted': item['text'], 'reference': record['target'], 'stopped_by': item['stopped_by']})
print({'new_sentences': {'verdict': new_report['verdict'], 'metrics': new_report['metrics'], 'reason': new_report['reason']}})
with open('outputs/marianmt_translation_translations.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['id', 'source', 'translation', 'reference', 'input_tokens', 'generated_tokens', 'stopped_by'])
    writer.writeheader()
    for record, item in zip(new_records, new_result['translations'], strict=True):
        writer.writerow({'id': record['id'], 'source': record['source'], 'translation': item['text'], 'reference': record['target'], 'input_tokens': item['input_tokens'], 'generated_tokens': item['generated_tokens'], 'stopped_by': item['stopped_by']})

artifact_dir = Path('outputs/marianmt_translation_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'marianmt_translation', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = MarianMTTranslationPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [item['text'] for item in pipe.translate([r['source'] for r in test_records[:8]], max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)['translations']]
after = [item['text'] for item in reloaded.translate([r['source'] for r in test_records[:8]], max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)['translations']]
parity = {'identical_translations': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_translations'] == parity['of']

weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'pytorch pickle, digest-verified, loaded with weights_only=True', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'url': CORPUS_URL, 'sha256': CORPUS_SHA256, 'license': CORPUS_LICENSE},
    'comparison': comparison,
    'new_sentences': new_report,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/marianmt_translation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen model already translates Tatoeba-style sentences well — its test BLEU sits where the upstream README puts it — and a bounded fine-tuning of the last two decoder blocks on 1,200 in-domain pairs still lifts held-out chrF by a few points and BLEU by several, in about a minute on CPU, with a 34 MB adapter that reloads to identical translations. That is the claim: the adaptation contract works end to end on a real parallel corpus, and the numbers it produces are read against a copy-source baseline and the frozen model rather than in isolation.

The test split is 300 sentences from one seeded split of one corpus, the metrics are two reference-based scores (own implementations, not sacrebleu-identical, and neither a human judgement), and Tatoeba is short, conversational and already in the model's training lineage. So a gain here says the contract works, not that the adapted model is better on your domain, that it handles long or technical text, or that its fluent output is faithful — a translation can drop a negation, change a number or leave an entity in English and still score well on character n-grams. Fine-tuning on a narrow corpus can also erode the model elsewhere; nothing here measures that.

Three things to carry to real data. **References first:** the copy-source baseline and the frozen model's score on *your* references are the two numbers to read before any adapted one. **Leakage:** de-duplicate sources across splits (the contract does this case-insensitively) and split by document or session when your pairs come from one, never at random over near-duplicates. **Ceilings:** inputs over `MAX_INPUT_TOKENS` are refused at inference and truncated to 128 pieces only during training — long-document translation is out of scope.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot (including its pickle weight file), fetch and digest-verify a real parallel corpus, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against a trivial baseline and the frozen model on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, translation quality on any other domain, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_DECODER_LAYERS = 6` to train the whole decoder and compare the artifact size and the test scores; raise `EPOCHS`; set `NUM_BEAMS = 1` and read the greedy scores; or bring your own pairs through BYOD and read the copy-source baseline before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/marianmt-en-tl-translation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/marianmt-en-tl-translation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance and the pickle trust boundary: https://github.com/kurtvalcorza/marianmt-en-tl-translation-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Helsinki-NLP/opus-mt-en-tl
- Upstream training code: https://github.com/Helsinki-NLP/OPUS-MT-train
- OPUS-MT — Building open translation services for the World (Tiedemann & Thottingal, EAMT 2020): https://aclanthology.org/2020.eamt-1.61
- Marian: Fast Neural Machine Translation in C++ (Junczys-Dowmunt et al., ACL 2018): https://arxiv.org/abs/1804.00344
- Tatoeba en–tl via OPUS (Tiedemann, LREC 2012; corpus release v2023-04-12, CC BY 2.0 FR): https://opus.nlpl.eu/Tatoeba-v2023-04-12.php
- chrF: character n-gram F-score for automatic MT evaluation (Popović, WMT 2015): https://aclanthology.org/W15-3049
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)